In [1]:
!ls -lh /content/combined_images.zip

ls: cannot access '/content/combined_images.zip': No such file or directory


In [2]:
!pip install -q kaggle

In [3]:
!pip install -q -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.1 MB/s eta 0:00:00


In [4]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle token: ")

Paste your Kaggle token: ··········


In [5]:
!kaggle datasets list -s "alzheimers multiclass"


ref                                                               title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
warcoder/alzheimers-disease-mri-images-dataset                    Alzheimers disease MRI images dataset            71187099  2023-09-11 15:11:15.480000            784         16  0.6875           
aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented  Alzheimer's Disease Multiclass Images Dataset   417170579  2024-06-26 23:19:46.027000           6767         33  0.75             
machine111sss/4class-alzheimers-mri-dataset                       "4class-alzheimers_mri_dataset"                  66272451  2026-06-01 14:59:38.093000             17          0  0.4375           


In [6]:
!kaggle datasets download -d aryan singhal10/alzheimers-multiclass-dataset-equal-and-augmented

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


In [7]:
!pip install -q -U kaggle

In [8]:
dataset = "aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented"

!kaggle datasets download -d {dataset} --unzip

Dataset URL: https://www.kaggle.com/datasets/aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented
License(s): apache-2.0
100% 398M/398M [00:06<00:00, 64.2MB/s]



In [13]:
import os

data_dir = "/content/combined_images"

classes = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented"
]

for cls in classes:
    path = os.path.join(data_dir, cls)
    count = len([
        f for f in os.listdir(path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    print(f"{cls}: {count} images")

MildDemented: 10000 images
ModerateDemented: 10000 images
NonDemented: 12800 images
VeryMildDemented: 11200 images


In [14]:
import os
import shutil
import random

source_dir = "/content/combined_images"
split_dir = "/content/alzheimer_split"

classes = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented"
]

# Create folders
for split in ["train", "val", "test"]:
    for cls in classes:
        os.makedirs(
            os.path.join(split_dir, split, cls),
            exist_ok=True
        )

random.seed(42)

for cls in classes:
    source_class_dir = os.path.join(source_dir, cls)

    images = [
        f for f in os.listdir(source_class_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)

    n = len(images)

    train_end = int(0.70 * n)
    val_end = train_end + int(0.15 * n)

    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    for img in train_images:
        shutil.copy2(
            os.path.join(source_class_dir, img),
            os.path.join(split_dir, "train", cls, img)
        )

    for img in val_images:
        shutil.copy2(
            os.path.join(source_class_dir, img),
            os.path.join(split_dir, "val", cls, img)
        )

    for img in test_images:
        shutil.copy2(
            os.path.join(source_class_dir, img),
            os.path.join(split_dir, "test", cls, img)
        )

    print(
        cls,
        "→",
        len(train_images), "train,",
        len(val_images), "val,",
        len(test_images), "test"
    )

MildDemented → 7000 train, 1500 val, 1500 test
ModerateDemented → 7000 train, 1500 val, 1500 test
NonDemented → 8960 train, 1920 val, 1920 test
VeryMildDemented → 7839 train, 1680 val, 1681 test


In [15]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_dir = "/content/alzheimer_split/train"
val_dir = "/content/alzheimer_split/val"
test_dir = "/content/alzheimer_split/test"

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Classes:", train_ds.class_names)

Found 30799 files belonging to 4 classes.
Found 6600 files belonging to 4 classes.
Found 6601 files belonging to 4 classes.
Classes: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']


In [16]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("Dataset pipeline ready!")

Dataset pipeline ready!


In [17]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Rescaling(1./127.5, offset=-1),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.2),

    layers.Dense(4, activation="softmax")
])

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [18]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

Model compiled successfully!


In [19]:
import time

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

training_time = time.time() - start_time

print(f"\nTraining completed in {training_time/60:.2f} minutes")

Epoch 1/5
963/963 ━━━━━━━━━━━━━━━━━━━━ 79s 63ms/step - accuracy: 0.5803 - loss: 0.9677 - val_accuracy: 0.6897 - val_loss: 0.7476
Epoch 2/5
963/963 ━━━━━━━━━━━━━━━━━━━━ 34s 35ms/step - accuracy: 0.6633 - loss: 0.7773 - val_accuracy: 0.6662 - val_loss: 0.7402
Epoch 3/5
963/963 ━━━━━━━━━━━━━━━━━━━━ 38s 39ms/step - accuracy: 0.6758 - loss: 0.7403 - val_accuracy: 0.7173 - val_loss: 0.6671
Epoch 4/5
963/963 ━━━━━━━━━━━━━━━━━━━━ 37s 35ms/step - accuracy: 0.6879 - loss: 0.7198 - val_accuracy: 0.7242 - val_loss: 0.6420
Epoch 5/5
963/963 ━━━━━━━━━━━━━━━━━━━━ 41s 35ms/step - accuracy: 0.6920 - loss: 0.7041 - val_accuracy: 0.7233 - val_loss: 0.6426

Training completed in 3.81 minutes


In [20]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("\n===== TEST RESULTS =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

207/207 ━━━━━━━━━━━━━━━━━━━━ 16s 79ms/step - accuracy: 0.7275 - loss: 0.6446

===== TEST RESULTS =====
Test Loss: 0.6446
Test Accuracy: 72.75%


In [22]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Get the true labels
y_true = np.concatenate([y.numpy() for x, y in test_ds])

# Get predictions
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

# We already know the correct class order from Step 8
class_names = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented"
]

# Classification report
print("===== CLASSIFICATION REPORT =====")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("\n===== CONFUSION MATRIX =====")
print(cm)

207/207 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step
===== CLASSIFICATION REPORT =====
                  precision    recall  f1-score   support

    MildDemented     0.5931    0.8453    0.6971      1500
ModerateDemented     0.9248    0.9913    0.9569      1500
     NonDemented     0.7397    0.6943    0.7163      1920
VeryMildDemented     0.6781    0.4247    0.5223      1681

        accuracy                         0.7275      6601
       macro avg     0.7339    0.7389    0.7231      6601
    weighted avg     0.7327    0.7275    0.7172      6601


===== CONFUSION MATRIX =====
[[1268   47   76  109]
 [   8 1487    3    2]
 [ 339   20 1333  228]
 [ 523   54  390  714]]


In [23]:
import time
import numpy as np

# Take one batch from the test dataset
images, labels = next(iter(test_ds))

# Warm-up prediction
model.predict(images, verbose=0)

# Measure prediction time
num_runs = 20
times = []

for _ in range(num_runs):
    start = time.perf_counter()

    model.predict(images, verbose=0)

    end = time.perf_counter()
    times.append(end - start)

avg_batch_time = np.mean(times)
avg_image_time = avg_batch_time / images.shape[0]

print("===== INFERENCE LATENCY =====")
print(f"Batch size: {images.shape[0]}")
print(f"Average batch inference time: {avg_batch_time * 1000:.2f} ms")
print(f"Average per-image inference time: {avg_image_time * 1000:.2f} ms")

===== INFERENCE LATENCY =====
Batch size: 32
Average batch inference time: 145.90 ms
Average per-image inference time: 4.56 ms


In [24]:
model.save("/content/alzheimer_mobilenetv2_baseline.keras")

print("Model saved successfully!")

Model saved successfully!


In [25]:
import os

model_path = "/content/alzheimer_mobilenetv2_baseline.keras"

print(f"File exists: {os.path.exists(model_path)}")
print(f"File size: {os.path.getsize(model_path) / (1024**2):.2f} MB")

File exists: True
File size: 9.23 MB


In [26]:
import json

metrics = {
    "model_name": "MobileNetV2",
    "model_version": "baseline_v1",
    "dataset": "Alzheimer's Multiclass Dataset",
    "num_classes": 4,
    "classes": [
        "MildDemented",
        "ModerateDemented",
        "NonDemented",
        "VeryMildDemented"
    ],
    "train_images": 30799,
    "validation_images": 6600,
    "test_images": 6601,
    "epochs": 5,
    "training_time_minutes": 3.81,
    "best_validation_accuracy": 0.7242,
    "test_accuracy": 0.7275,
    "macro_f1": 0.7231,
    "weighted_f1": 0.7172,
    "inference_latency_ms_per_image": 4.56,
    "hardware": "Google Colab NVIDIA T4"
}

with open("/content/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("metrics.json created!")

metrics.json created!


In [28]:
import numpy as np

# Generate probability predictions for the test set
y_prob = model.predict(test_ds, verbose=1)

print("Prediction shape:", y_prob.shape)
print("First prediction:", y_prob[0])
print("Probability sum:", np.sum(y_prob[0]))

207/207 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step
Prediction shape: (6601, 4)
First prediction: [0.92008054 0.00410452 0.02115674 0.05465818]
Probability sum: 0.99999994


In [29]:
np.save("/content/test_predictions.npy", y_prob)

print("Predictions saved successfully!")

Predictions saved successfully!
